In [ ]:
import pandas as pd
from keras.models import Sequential
from keras.layers import LSTM, Dense, Dropout, BatchNormalization
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_absolute_error, r2_score
import matplotlib.pyplot as plt
import numpy as np
import os

In [ ]:
# The heart of this notebook, the LSTM.
class MyLSTM():
    def __init__(self):
        # The LSTM architecture.
        self.model = Sequential([
            LSTM(64, return_sequences=True, input_shape=(24, 22), recurrent_dropout=0.1),
            Dropout(0.2),
            LSTM(32),
            Dropout(0.2),
            Dense(16, activation='relu'),
            BatchNormalization(),
            Dense(1)
        ])

        self.model.compile(
            optimizer='adam', 
            loss='mean_squared_error'
        )

    # Trains the model.
    def Train(self, X_train, y_train, dates_train, epochs, batch_size, validation_split):
        self.X_train = X_train
        self.y_train = y_train
        self.dates_train = dates_train
        self.training_data = self.model.fit(X_train, 
                                            y_train, 
                                            epochs=epochs, 
                                            batch_size=batch_size, 
                                            validation_split=validation_split
                                            )
    
    # Tests the model on specified training data. It also reverses the scaling used in training 
    # in preparation for calculating error metrics.
    def Test(self, X_test, y_test, dates_test, scaler, features):
        self.X_test = X_test
        self.y_test = y_test
        self.dates_test = dates_test
        self.predictions = self.model.predict(X_test)
        
        y_true = self.y_test.reshape(-1,1)
        y_pred = self.predictions.reshape(-1,1)

        dummy_true = np.zeros((len(y_true), len(features)))
        dummy_pred = np.zeros((len(y_pred), len(features)))

        dummy_true[:,0] = y_true[:,0]
        dummy_pred[:,0] = y_pred[:,0]

        self.y_test = scaler.inverse_transform(dummy_true)[:,0]
        self.predictions = scaler.inverse_transform(dummy_pred)[:,0]
    
    # Plots the predicted demand against the actual demand for a single week in October 2022.
    def VisualizeActualVsPredicted(self, title):
        plt.figure(figsize=(12, 6))
        plt.plot(self.dates_test[:168], self.y_test[:168], label="Actual Demand")
        plt.plot(self.dates_test[:168], self.predictions[:168], label="Predicted Demand")
        plt.title(title)
        plt.xlabel('Date')
        plt.ylabel('Demand (Passengers)')
        plt.legend()
        os.makedirs('../../Graphs', exist_ok=True)
        plt.savefig('../../Graphs/lstm_actual_vs_pred.jpeg')
        plt.show()

    # Generates a loss curve.
    def VisualizeLoss(self, title):
        plt.figure(figsize=(12, 6))
        plt.plot(self.training_data.history['loss'], label='Training Loss')
        plt.plot(self.training_data.history['val_loss'], label='Validation Loss')
        plt.title(title)
        plt.xlabel('Epoch')
        plt.ylabel('Loss (MSE)')
        plt.legend()
        os.makedirs('../../Graphs', exist_ok=True)
        plt.savefig('../../Graphs/lstm_loss.jpeg')
        plt.show()

    # Generates a CSV file containing each prediction compared to the actual value for that hour.
    def WritePredsToCSV(self):
        output = pd.DataFrame({
            'datetime': self.dates_test,
            'actual_demand': self.y_test.flatten(),
            'predicted_demand': self.predictions.flatten()
        })
        os.makedirs('../../Results', exist_ok=True)
        output.to_csv('../../Results/actual_vs_predicted_lstm.csv', index=False)
    
    # Prints the RMSE, MAE and R^2 Score.
    def PrintMetrics(self):
        self.rmse = np.sqrt(np.mean((self.y_test - self.predictions)**2))
        self.mae = mean_absolute_error(self.y_test, self.predictions)
        self.r2 = r2_score(self.y_test, self.predictions)

        print(f"RMSE: {self.rmse}")
        print(f"MAE: {self.mae}")
        print(f"R^2 Score: {self.r2}")

In [ ]:
# Organizes the data into sequences of a chosen length.
def create_sequences(X_scaled, y_scaled, dates, lookback=24): 
    X = []
    y = []
    sequence_dates = []

    for i in range(lookback, len(X_scaled)):
        X.append(X_scaled[i-lookback:i])
        y.append(y_scaled[i][0])
        sequence_dates.append(dates.index[i])
    
    return np.array(X), np.array(y), np.array(sequence_dates)

In [ ]:
# Loads the file in.
data = pd.read_csv('../../Data/data_with_weather.csv')

In [ ]:
# Transforms the dates into datetime objects.
data['datetime'] = pd.to_datetime(data['datetime'])
data.set_index('datetime', inplace=True)

hours = data.index.hour
dow = data.index.dayofweek

# We engineer various additional features here.
data['is_morning_peak'] = ((hours >= 7) & (hours <= 9)).astype(int)
data['is_midday'] = ((hours >= 11) & (hours <= 14)).astype(int)
data['is_evening_peak'] = ((hours >= 16) & (hours <= 18)).astype(int)
data['is_late_night'] = ((hours >= 22) | (hours <= 5)).astype(int)
data['hour_sin'] = np.sin(2 * np.pi * hours / 24)
data['hour_cos'] = np.cos(2 * np.pi * hours / 24)
data['dow_sin'] = np.sin(2 * np.pi * dow / 7)
data['dow_cos'] = np.cos(2 * np.pi * dow / 7)
data['bad_weather'] = (data['is_snowing'] | data['is_raining']).astype(int)
data['feels_cold'] = (data['feelslike'] <= 32).astype(int)
data['feels_very_cold'] = (data['feelslike'] <= 20).astype(int)
data['cold_and_snowing'] = (data['is_snowing'] & data['feels_cold']).astype(int)
demand = data['demand'].astype(float).values.reshape(-1, 1)

features = [
    'demand',
    'is_weekend',
    'event',
    'temp',
    'feelslike',
    'precip',
    'snow',
    'snowdepth',
    'is_raining',
    'is_snowing',
    'is_morning_peak',
    'is_midday',
    'is_evening_peak',
    'is_late_night',
    'hour_sin',
    'hour_cos',
    'dow_sin',
    'dow_cos',
    'bad_weather',
    'feels_cold',
    'feels_very_cold',
    'cold_and_snowing'
]

In [ ]:
# Our train/test split occurs here.
split_idx = int(len(data) * 0.8)
train_df = data.iloc[:split_idx]
test_df  = data.iloc[split_idx:]

# Now we scale everything.
X_scaler = MinMaxScaler()
y_scaler = MinMaxScaler()

X_train_df = train_df[features]
y_train_df = train_df[['demand']]
X_test_df = test_df[features]
y_test_df = test_df[['demand']]

X_scaler.fit(X_train_df)
y_scaler.fit(y_train_df)

X_train_scaled = X_scaler.transform(X_train_df)
X_test_scaled  = X_scaler.transform(X_test_df)

y_train_scaled = y_scaler.transform(y_train_df)
y_test_scaled  = y_scaler.transform(y_test_df)

# The sequences are assembled here.
X_train, y_train, dates_train = create_sequences(X_train_scaled, y_train_scaled, train_df)
X_test, y_test, dates_test = create_sequences(X_test_scaled, y_test_scaled, test_df)

In [ ]:
# Now we finally train and test the model.
model = MyLSTM()
model.Train(X_train, y_train, dates_train, 250, 32, 0.1)
model.Test(X_test, y_test, dates_test, y_scaler, features)

In [ ]:
model.VisualizeActualVsPredicted("Actual Demand vs. Predicted Demand (LSTM)")

In [ ]:
model.VisualizeLoss("LSTM Training & Validation Loss")

In [ ]:
model.WritePredsToCSV()

In [ ]:
model.PrintMetrics()